# Writing to Delta Lake

Delta is a transactional table format on top of Parquet — it adds ACID, time travel, MERGE/UPDATE/DELETE, and schema evolution.

**Compatibility note**: this image runs Spark **4.1.3 / Scala 2.13**, so we use `delta-spark_2.13:4.3.1`. Picking the wrong Delta version for your Spark version is the #1 cause of cryptic class-not-found errors.

The first run downloads the jar via `spark.jars.packages` — that takes a moment. Subsequent runs use the cached jar.

In [1]:
# Optional: install the Python API (DeltaTable). Skip if you only use SQL.
# ! pip install delta-spark==4.3.1

In [2]:
from pyspark.sql import SparkSession, functions as F

In [3]:
spark = (
    SparkSession.builder
    .appName("Write Delta")
    .master("local[2]")
    .config("spark.jars.packages", "io.delta:delta-spark_2.13:4.3.1")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    .getOrCreate()
)

:: loading settings :: url = jar:file:/opt/spark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /root/.ivy2/cache
The jars for the packages stored in: /root/.ivy2/jars
io.delta#delta-spark_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-13ad13bb-d345-42e6-a151-b6bc38e97fe8;1.0
	confs: [default]
	found io.delta#delta-spark_2.12;3.2.0 in central
	found io.delta#delta-storage;3.2.0 in central
	found org.antlr#antlr4-runtime;4.9.3 in central
downloading https://repo1.maven.org/maven2/io/delta/delta-spark_2.12/3.2.0/delta-spark_2.12-3.2.0.jar ...
	[SUCCESSFUL ] io.delta#delta-spark_2.12;3.2.0!delta-spark_2.12.jar (468ms)
downloading https://repo1.maven.org/maven2/io/delta/delta-storage/3.2.0/delta-storage-3.2.0.jar ...
	[SUCCESSFUL ] io.delta#delta-storage;3.2.0!delta-storage.jar (35ms)
downloading https://repo1.maven.org/maven2/org/antlr/antlr4-runtime/4.9.3/antlr4-runtime-4.9.3.jar ...
	[SUCCESSFUL ] org.antlr#antlr4-runtime;4.9.3!antlr4-runtime.jar (55ms)
:: resolution report :: resolve 2527ms :: artifacts dl 56

In [5]:
! curl -o /opt/examples/datasets/Mall_Customers.csv \
https://raw.githubusercontent.com/erkansirin78/datasets/master/Mall_Customers.csv

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100  4365  100  4365    0     0  19507      0 --:--:-- --:--:-- --:--:-- 19573


In [6]:
df = spark.read \
    .option("header", True) \
    .option("inferSchema", True) \
    .csv("file:///opt/examples/datasets/Mall_Customers.csv")

df.printSchema()
df.limit(5).toPandas()

root
 |-- CustomerID: integer (nullable = true)
 |-- Gender: string (nullable = true)
 |-- Age: integer (nullable = true)
 |-- AnnualIncome: integer (nullable = true)
 |-- SpendingScore: integer (nullable = true)



,CustomerID,Gender,Age,AnnualIncome,SpendingScore
0,1,Male,19,15000,39
1,2,Male,21,15000,81
2,3,Female,20,16000,6
3,4,Female,23,16000,77
4,5,Female,31,17000,40


## 1. Write — basic

In [7]:
delta_path = "file:///tmp/pyspark_output_data/mall_delta"

df.write \
    .format("delta") \
    .mode("overwrite") \
    .save(delta_path)

In [8]:
# Notice the _delta_log/ folder — that's the transaction log
! ls -lh /tmp/pyspark_output_data/mall_delta
! ls -lh /tmp/pyspark_output_data/mall_delta/_delta_log

total 8.0K
drwxr-xr-x 3 root root 4.0K Apr 25 02:26 _delta_log
-rw-r--r-- 1 root root 3.7K Apr 25 02:26 part-00000-59cfe3db-4c40-44d5-97c2-8244ceda7ac3-c000.snappy.parquet
total 8.0K
-rw-r--r-- 1 root root 1.6K Apr 25 02:26 00000000000000000000.json
drwxr-xr-x 2 root root 4.0K Apr 25 02:26 _commits


## 2. Read it back

In [9]:
df_read = spark.read.format("delta").load(delta_path)
df_read.count()

26/04/25 02:26:41 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


200

## 3. Partitioned write

In [10]:
df.write \
    .format("delta") \
    .mode("overwrite") \
    .partitionBy("Gender") \
    .save("file:///tmp/pyspark_output_data/mall_delta_partitioned")

In [11]:
! ls /tmp/pyspark_output_data/mall_delta_partitioned

'Gender=Female'  'Gender=Male'	 _delta_log


## 4. Append

In [12]:
# Take 5 rows and append them to simulate new data
df_more = df.limit(5)
df_more.write.format("delta").mode("append").save(delta_path)

spark.read.format("delta").load(delta_path).count()

205

## 5. Time travel

Every write is a numbered version. You can read any past version.

In [13]:
# version 0 = first overwrite, version 1 = the append above
spark.read.format("delta").option("versionAsOf", 0).load(delta_path).count()

200

In [14]:
spark.read.format("delta").option("versionAsOf", 1).load(delta_path).count()

205

## 6. Update / Delete via SQL

In [15]:
spark.sql(f"""
    UPDATE delta.`{delta_path}`
    SET SpendingScore = SpendingScore + 1
    WHERE Gender = 'Female'
""")

DataFrame[num_affected_rows: bigint]

In [16]:
spark.sql(f"""
    DELETE FROM delta.`{delta_path}`
    WHERE Age > 65
""")

DataFrame[num_affected_rows: bigint]

## 7. History

In [17]:
spark.sql(f"DESCRIBE HISTORY delta.`{delta_path}`").show(truncate=False)

+-------+-----------------------+------+--------+---------+-----------------------------------------+----+--------+---------+-----------+--------------+-------------+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+------------+-----------------------------------+
|version|timestamp              |userId|userName|operation|operationParameters                      |job |notebook|clusterId|readVersion|isolationLevel|isBlindAppend|operationMetrics                                                                                                                                                                                                                                                                                        

In [18]:
spark.stop()